1. Setup

In [1]:
import os
import sys
 
PROJECT_ROOT ="/Users/Anna/ecommerce-data-pipeline"

os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("cwd", os.getcwd())

cwd /Users/Anna/ecommerce-data-pipeline


In [2]:
from src.transformation import(
    load_clean_data,
    transform_data
)

from src.rfm_features import compute_rfm
from src.churn_target import create_churn_target

from config.paths import CLEAN_RETAIL

2. Loading and transforming data

In [3]:
df = load_clean_data(CLEAN_RETAIL)

df = transform_data(df)

df.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,order_value,invoice_year,invoice_month,invoice_hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,12,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,8


3. Define temporal cutoff

In [4]:
import pandas as pd

max_date = df["invoice_date"].max()

cutoff_date = max_date - pd.Timedelta(days=90)

print(max_date)
print(cutoff_date)

2011-12-09 12:50:00
2011-09-10 12:50:00


4. Create feature and target datasets

In [5]:
feature_df = df[
    df["invoice_date"] < cutoff_date
]

future_df = df[
    df["invoice_date"] >= cutoff_date
]

In [6]:
feature_df.shape


(236379, 12)

In [7]:
future_df.shape

(161545, 12)

5. Generate RFM features

In [8]:
rfm_df = compute_rfm(feature_df)

rfm_df.head()

,customer_id,recency,frequency,avg_order_value
0,12346.0,234,1,77183.60
1,12347.0,38,5,558.17
2,12348.0,157,3,495.75
3,12350.0,218,1,334.40
4,12352.0,170,5,312.36


6. Generate churn target

In [9]:
target_df = create_churn_target(
    feature_df,
    future_df
)

target_df

,customer_id,churn
0,17850.0,1
9,13047.0,0
26,12583.0,0
46,13748.0,1
65,15100.0,1
...,...,...
235704,16603.0,0
236031,14824.0,0
236046,16910.0,0
236099,15652.0,1


In [10]:
target_df["churn"].value_counts(normalize=True)

churn
0    0.57003
1    0.42997
Name: proportion, dtype: float64

7. Create training dataset - first supervised-learning dataset

In [11]:
training_df = rfm_df.merge(
    target_df,
    on="customer_id",
    how="inner"
)

8. Exploratory analysis

In [12]:
training_df.groupby("churn").mean()

,customer_id,recency,frequency,avg_order_value
churn,,,,
0,15249.991671,72.673087,4.799063,396.078704
1,15310.766046,122.006211,1.855763,401.580324


9. Train model

In [ ]:
X = training_df[
    ["recency", "frequency", "avg_order_value"]
]

y = training_df["churn"]

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
   X,
   y,
   test_size=0.2, 
   random_state=42,
   stratify=y
)

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [16]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    random_state=42
)

model.fit(
    X_train_scaled,
    y_train
)

y_pred = model.predict(X_test_scaled)

10. Evaluation

In [17]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(accuracy)

0.6602373887240356


In [18]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.70      0.70      0.70       384
           1       0.60      0.61      0.61       290

    accuracy                           0.66       674
   macro avg       0.65      0.65      0.65       674
weighted avg       0.66      0.66      0.66       674



In [19]:
import pandas as pd

pd.DataFrame({
    "feature":X.columns,
    "coefficient":model.coef_[0]
})

,feature,coefficient
0,recency,0.320149
1,frequency,-2.047834
2,avg_order_value,0.010381


## Interpretation of Temporal Churn Experiment

The initial 99% accuracy was largely driven by recency, which closely reflected the target definition. After introducing a temporal split to eliminate this information advantage, model accuracy dropped to 66%.

Feature coefficients indicate that purchase frequency became the dominant predictor of future churn, while average order value contributed little predictive signal.

This suggests that customer loyalty, measured through historical purchase frequency, is a stronger indicator of future retention than spending level. The temporal experiment also demonstrates the importance of preventing information leakage when evaluating churn models.